# 06 — Final Results

**Objective:** summarise the out-of-sample performance and the main conclusion from the research.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..")
RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

returns = pd.read_csv(
    RESULTS_DIR / "oos_returns.csv",
    index_col=0,
    parse_dates=True
).squeeze("columns")

returns = pd.to_numeric(returns, errors="coerce").dropna()

## Performance metrics

In [ ]:
equity = (1 + returns).cumprod()

total_return = equity.iloc[-1] - 1
annualised_return = (1 + total_return) ** (252 / len(returns)) - 1
annualised_volatility = returns.std() * np.sqrt(252)
sharpe_ratio = annualised_return / annualised_volatility

drawdown = equity / equity.cummax() - 1
max_drawdown = drawdown.min()

active_returns = returns[returns != 0]
win_rate = (active_returns > 0).mean()

metrics = pd.DataFrame({
    "Metric": [
        "Total Return",
        "Annualised Return",
        "Annualised Volatility",
        "Sharpe Ratio",
        "Maximum Drawdown",
        "Active-Day Win Rate"
    ],
    "Value": [
        total_return,
        annualised_return,
        annualised_volatility,
        sharpe_ratio,
        max_drawdown,
        win_rate
    ]
})

metrics

## Equity curve

In [ ]:
equity.plot(figsize=(10, 4))
plt.title("Out-of-Sample Equity Curve")
plt.xlabel("Date")
plt.ylabel("Growth of 1 unit")
plt.grid(alpha=0.3)
plt.show()

## Drawdown

In [ ]:
drawdown.plot(figsize=(10, 4))
plt.title("Out-of-Sample Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.grid(alpha=0.3)
plt.show()

## Conclusion

The baseline KO/PEP strategy did not produce positive out-of-sample performance. In my final run, the strategy returned approximately **-8.5%** overall, with an annualised return of **-1.5%**, a Sharpe ratio of **-0.16**, and a maximum drawdown of approximately **-19.3%**.

The main result is therefore methodological rather than a claim of profitability. The project shows how an apparently reasonable statistical relationship can weaken once it is evaluated with rolling parameter estimation, unseen test periods and transaction costs.

The next improvements I would investigate are broader pair selection, more stable hedge-ratio estimation, stricter formation-period filters and alternative position-sizing rules.

## Project summary

I built a statistical-arbitrage research pipeline in Python using cointegration, regression-based hedge ratios and rolling z-score signals. I then evaluated the strategy with walk-forward out-of-sample testing, transaction costs and parameter-sensitivity analysis to reduce look-ahead bias and assess robustness.